# 第 53 课：OAuth 授权码、PKCE 与 MCP 资源访问

**同意页 → 授权码 → 换取访问令牌 → 资源服务验证权限。每一步都有不同的职责。**

上一课的开发凭证由宿主预先发给客户端。本课增加一次可观察的授权流程，让客户端在批准后自己获得短期、限资源和限权限的访问令牌。

默认只用 **Python 3.10+ 标准库**，不下载模型、不调用外部服务、不使用 GPU。Notebook 与同名 `.py` 各自独立，可以从新内核顺序运行。

**实验边界：** 三个不同端口的回环 HTTP 监听器实际运行，但都在同一进程的线程中。授权页面固定显示模拟 Alice，没有验证密码、会话登录或真实账号。默认批准是脚本化浏览器往返，不是真人审阅成绩。全链路 HTTP、共享内存 token 验证器和有限工具子集均用于教学，不能作为生产 OAuth 服务部署。

普通阅读可分两次：先做 PKCE 与回调实验，再观察真实 HTTP 往返、测试和练习。

## 学习目标与材料结构

学完后应能解释：为什么拿到 code 仍不能直接访问工具；`state`、`code_verifier`、`iss`、`scope` 与 `resource` 分别检查什么；为什么已获得只读令牌也不能调用计算器；用户拒绝和授权失败有什么区别。

讲义把完整实现分成可运行的定义单元，再用短实验调用它们。较长的服务实现和检查单元默认可折叠，先运行、观察数据流，再回看函数内部。独立练习默认关闭，运行全部单元成功不等于完成练习。

运行这些实验会在 `127.0.0.1` 上临时监听三个随机端口，并在实验结束后关闭。受限环境不允许监听时，会明确报错，不回退成伪造网络结果。

## 1．先核对版本：OAuth 2.1 仍是草案

截至 **2026-11-02**，IETF 的 OAuth 2.1 文档为 `draft-ietf-oauth-v2-1-17`，状态仍是 Internet-Draft，不是已发布的 OAuth 2.1 RFC。[R1]

本课学习授权码与 S256 PKCE，结合已经发布的 RFC 7636、8414、8707、9207、9700。[R2–R6] 不实现 implicit、密码授权、refresh token、动态客户端注册或 OpenID Connect。

一个细节：OAuth 2.1 草案已移除**换码请求**中的 `redirect_uri` 参数；它仍然属于**授权请求**。本课主路径不在 `/token` 发送它；若兼容客户端提供了该参数，本课会核对它与原请求完全一致。[R1 §10.2]

这不是“回调地址无需校验”。授权入口依然先核对登记的回调目标，错误时不重定向。

In [1]:
from __future__ import annotations

# %% 01 基础约定、PKCE 和严格解析
import argparse
import base64
import copy
import hashlib
import hmac
import html
import http.client
import json
import platform
import re
import secrets
import sys
import threading
import time
from contextlib import AbstractContextManager
from dataclasses import dataclass, field
from datetime import datetime, timezone
from html.parser import HTMLParser
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer
from pathlib import Path
from urllib.parse import parse_qsl, urlencode, urlsplit

VERSION = "lesson53-code-pkce-v1"
PROTOCOL = "2026-07-28"
META = "io.modelcontextprotocol/"
CLIENT_ID = "lesson53-public-client"
SCOPES = frozenset({"docs:read", "calc:use"})
MAX_BODY = 32768
RESOURCE_META = "/.well-known/oauth-protected-resource/mcp"
AS_META = "/.well-known/oauth-authorization-server"
VERIFIER_RE = re.compile(r"[A-Za-z0-9._~-]{43,128}\Z")
RFC_VERIFIER = "dBjftJeZ4CVP-mB92K27uhbUJU1p1r_wW1gFWFOEjXk"
RFC_CHALLENGE = "E9Melhoa2OwvFrEMTJguCHaoeK1t8URWbuGJSstw-cM"
SOURCES = {
    "oauth21_status": "https://datatracker.ietf.org/doc/draft-ietf-oauth-v2-1/",
    "oauth21_fixed": "https://www.ietf.org/ietf-ftp/internet-drafts/draft-ietf-oauth-v2-1-17.html",
    "pkce": "https://www.rfc-editor.org/rfc/rfc7636.html",
    "security": "https://www.rfc-editor.org/rfc/rfc9700.html",
    "native": "https://www.rfc-editor.org/rfc/rfc8252.html",
    "as_metadata": "https://www.rfc-editor.org/rfc/rfc8414.html",
    "resource": "https://www.rfc-editor.org/rfc/rfc8707.html",
    "issuer": "https://www.rfc-editor.org/rfc/rfc9207.html",
    "resource_metadata": "https://www.rfc-editor.org/rfc/rfc9728.html",
    "mcp": "https://modelcontextprotocol.io/specification/2026-07-28/basic/authorization",
    "http_server": "https://docs.python.org/3/library/http.server.html",
}

class LessonError(ValueError):
    pass

class ProtocolError(LessonError):
    def __init__(self, error: str, status: int = 400):
        super().__init__(error)
        self.error, self.status = error, status


def require(ok, message: str) -> None:
    if not ok:
        raise LessonError(message)


def canonical(value) -> str:
    return json.dumps(value, ensure_ascii=False, sort_keys=True,
                      separators=(",", ":"), allow_nan=False)


def sha(value: str) -> str:
    return hashlib.sha256(value.encode("utf-8")).hexdigest()


def strict_json(text: str):
    def pairs(items):
        out = {}
        for key, value in items:
            require(key not in out, "重复 JSON 键")
            out[key] = value
        return out
    def bad_constant(_):
        raise LessonError("非有限 JSON 值")
    return json.loads(text, object_pairs_hook=pairs, parse_constant=bad_constant)


def strict_form(text: str) -> dict[str, str]:
    require(len(text.encode("utf-8")) <= MAX_BODY, "表单过长")
    require(not re.search(r"%(?![0-9A-Fa-f]{2})", text), "错误的百分号编码")
    out = {}
    for key, value in parse_qsl(text, keep_blank_values=True, strict_parsing=True,
                                encoding="utf-8", errors="strict", max_num_fields=20):
        require(key not in out, "重复表单参数")
        out[key] = value
    return out


def pkce_s256(verifier: str) -> str:
    """RFC7636: BASE64URL(SHA256(ASCII(verifier)))，不是 hexdigest 后再编码。"""
    require(isinstance(verifier, str) and VERIFIER_RE.fullmatch(verifier),
            "verifier 必须为 43–128 个允许的 ASCII 字符")
    digest = hashlib.sha256(verifier.encode("ascii")).digest()
    return base64.urlsafe_b64encode(digest).rstrip(b"=").decode("ascii")


def new_verifier() -> str:
    return secrets.token_urlsafe(32)  # 256 位随机性，通常为 43 个字符。


def parse_scope(text: str) -> frozenset[str]:
    require(isinstance(text, str) and 0 < len(text) < 200, "scope 类型或长度错误")
    parts = text.split(" ")
    require(all(parts) and len(parts) == len(set(parts)), "scope 不能为空或重复")
    require(set(parts) <= SCOPES, "未知 scope")
    return frozenset(parts)


def loopback(url: str):
    p = urlsplit(url)
    require(p.scheme == "http" and p.hostname == "127.0.0.1" and p.port is not None,
            "本课只访问显式 127.0.0.1 HTTP 端点")
    require(not p.username and not p.password and not p.fragment, "URL 身份或片段不允许")
    return p

## 2．四个参与者，三组监听器

```text
资源所有者：模拟 Alice，在本地页面批准或拒绝
        ↓
公共客户端：本机 Python 程序，保留本次 verifier 和 state
        ↓                  ↖ 授权码回调 /callback
授权服务器 AS：/authorize → /consent → /token
        ↓ 签发适用于目标资源的访问令牌
资源服务器 RS：/mcp，验证令牌与操作权限
```

`client_id` 是公开标识，不是客户端密码。CLI/桌面公共客户端不能通过在源码里写一个固定 `client_secret` 证明自己拥有不可复制的秘密；PKCE 不把公共客户端变成保密客户端。[R1][R7]

OAuth 解决授权，不自动完成用户登录。本课用固定 Alice 替代身份认证系统，**不会从用户输入的 `subject=admin` 推断权限**。正式登录通常还涉及可信会话、MFA 或 OIDC 等独立机制。

AS 与 RS 用不同端口区分协议角色，但 RS 通过同一进程内的可信验证函数查询 opaque token 状态。没有实现网络 introspection，也不是两个独立安全域。

## 3．PKCE：先留下挑战，换码时证明自己持有 verifier

客户端为**每一次授权**生成新的随机 `code_verifier`：

```python
verifier = secrets.token_urlsafe(32)
challenge = BASE64URL(SHA256(ASCII(verifier)))
```

使用 SHA256 的原始 32 字节摘要，做 URL-safe Base64，去掉尾部 `=`。不是把 `hexdigest()` 的 64 个文字字符再编码，也不是可逆加密。[R2]

授权请求只发送 `challenge` 和 `code_challenge_method=S256`。客户端保留 verifier；换码 POST 才提交 verifier，AS 重新计算并比较。随机秘密通常是 43 个字符，合法长度范围为 43–128。[R2]

**下面是 RFC 公开测试向量，不是本次授权的秘密。**

In [2]:
print("RFC verifier（公开示例）：", RFC_VERIFIER)
print("S256 challenge：", pkce_s256(RFC_VERIFIER))
assert pkce_s256(RFC_VERIFIER) == RFC_CHALLENGE
sample = new_verifier()
print("本次新 verifier 的长度：", len(sample))
print("是否使用允许的字符：", bool(VERIFIER_RE.fullmatch(sample)))
del sample  # 不打印实际随机秘密。

RFC verifier（公开示例）： dBjftJeZ4CVP-mB92K27uhbUJU1p1r_wW1gFWFOEjXk
S256 challenge： E9Melhoa2OwvFrEMTJguCHaoeK1t8URWbuGJSstw-cM
本次新 verifier 的长度： 43
是否使用允许的字符： True


### 为什么 state 与 PKCE 同时存在？

| 值 | 本课用途 | 不是 |
|---|---|---|
| `state` | 客户端把回调关联到待处理授权，拒绝串线与重复回调 | 访问权限 |
| `code_verifier` | 证明换码者持有发起授权时保留的本次秘密 | 用户密码 |
| `code_challenge` | verifier 的 S256 摘要，绑定授权码 | 访问令牌 |
| `iss` | 回调来自哪个预期授权服务器 | 服务自报名字 |
| `resource` | 请求给哪个资源使用的令牌 | 可以任意调用的 URL |
| `scope` | 请求允许哪些类别的操作 | 所有对象的管理员权限 |

安全配置下 PKCE 也可以提供与 CSRF 相关的保护，不能把“PKCE 只处理一种攻击、绝不处理其他问题”背成绝对规则。本课同时校验随机 state、issuer 和 PKCE，便于分层观察。[R3][R6]

HTTP Web 客户端还需要把待处理状态与用户浏览器会话可靠绑定。本课是单用户、本机公共客户端，回调只关联宿主内存中当前启动的授权，不是多用户 Web 会话实现。

In [3]:
# 同一个 verifier 总是得到同一个 challenge；不同请求不能复用固定 verifier。
v1, v2 = new_verifier(), new_verifier()
print("两次新授权 verifier 不同：", v1 != v2)
print("同一 verifier 计算稳定：", pkce_s256(v1) == pkce_s256(v1))
print("challenge 长度：", len(pkce_s256(v1)))
del v1, v2

两次新授权 verifier 不同： True
同一 verifier 计算稳定： True
challenge 长度： 43


## 4．AS 状态：用户决定、授权码和 token 分别保存

授权页展示客户端、目标资源、请求 scope 和回调目标。批准表单绑定一次性票据、CSRF 随机值和 HttpOnly Cookie，并核对 Origin；这些措施并不补足缺失的真实用户登录。

授权码只保留有限时间，绑定 client、回调、资源、scope 和挑战；令牌绑定 subject、issuer、audience、scope 和过期时间。它们不是同一条记录。

本课批准时才创建授权码，拒绝时回调 `error=access_denied`。所有错误回调也要验证 state 与 issuer。[R6]

服务端主索引使用随机凭证的摘要；`secret_values` 是只在内存中的测试用秘密集合，用来检查导出报告没有泄漏，不导出、不用于生产。

In [4]:
# %% 02 授权服务器的内存状态：批准、一次性授权码与访问令牌
@dataclass
class AdjustableClock:
    offset: float = 0.0
    def now(self) -> float:
        return time.monotonic() + self.offset
    def advance(self, seconds: float) -> None:
        require(seconds >= 0, "时钟不能倒退")
        self.offset += seconds

@dataclass
class Consent:
    params: dict
    cookie_digest: str
    csrf_digest: str
    expires: float
    consumed: bool = False

@dataclass
class CodeRecord:
    client_id: str
    redirect_uri: str
    resource: str
    scopes: frozenset[str]
    challenge: str
    expires: float
    used: bool = False
    token_digests: list[str] = field(default_factory=list)

@dataclass
class TokenRecord:
    subject: str
    issuer: str
    audience: str
    scopes: frozenset[str]
    expires: float
    revoked: bool = False

class Authority:
    """AS 内存记录。RS 通过可信内存验证器查验 opaque token；不做网络 introspection。"""
    def __init__(self, clock: AdjustableClock | None = None):
        self.clock = clock or AdjustableClock()
        self.issuer = self.resource = self.redirect_uri = ""
        self.lock = threading.RLock()
        self.consents: dict[str, Consent] = {}
        self.codes: dict[str, CodeRecord] = {}
        self.tokens: dict[str, TokenRecord] = {}
        self.secret_values: list[str] = []  # 仅供本课验证日志不泄漏，不导出。
        self.counts = {"approved": 0, "denied": 0, "token_requests": 0,
                       "tokens_issued": 0, "resource_executions": 0}
        self.code_ttl, self.token_ttl, self.consent_ttl = 60, 120, 300
        self.cookie_name = "lesson53_" + secrets.token_hex(4)

    def secret(self, n=32) -> str:
        value = secrets.token_urlsafe(n)
        self.secret_values.append(value)
        return value

    def metadata(self) -> dict:
        return {
            "issuer": self.issuer,
            "authorization_endpoint": self.issuer + "/authorize",
            "token_endpoint": self.issuer + "/token",
            "response_types_supported": ["code"],
            "grant_types_supported": ["authorization_code"],
            "token_endpoint_auth_methods_supported": ["none"],
            "code_challenge_methods_supported": ["S256"],
            "authorization_response_iss_parameter_supported": True,
            "scopes_supported": sorted(SCOPES),
        }

    def validate_authorization(self, params: dict) -> frozenset[str]:
        expected = {"response_type", "client_id", "redirect_uri", "scope", "state",
                    "resource", "code_challenge", "code_challenge_method"}
        if set(params) != expected:
            raise ProtocolError("invalid_request")
        # 未确认回调目标之前，不重定向错误，防止开放重定向。
        if params["client_id"] != CLIENT_ID or params["redirect_uri"] != self.redirect_uri:
            raise ProtocolError("invalid_request")
        if params["response_type"] != "code":
            raise ProtocolError("unsupported_response_type")
        if params["resource"] != self.resource:
            raise ProtocolError("invalid_target")
        if params["code_challenge_method"] != "S256":
            raise ProtocolError("invalid_request")
        if not re.fullmatch(r"[A-Za-z0-9_-]{43}", params["code_challenge"]):
            raise ProtocolError("invalid_request")
        if not re.fullmatch(r"[A-Za-z0-9_-]{32,128}", params["state"]):
            raise ProtocolError("invalid_request")
        try:
            return parse_scope(params["scope"])
        except LessonError:
            raise ProtocolError("invalid_scope") from None

    def begin_consent(self, params: dict) -> tuple[str, str, str]:
        self.validate_authorization(params)
        with self.lock:
            require(len(self.consents) < 256, "本课内存许可目录已满")
            ticket, csrf, cookie = self.secret(), self.secret(), self.secret()
            self.consents[sha(ticket)] = Consent(copy.deepcopy(params), sha(cookie), sha(csrf),
                                                 self.clock.now() + self.consent_ttl)
        return ticket, csrf, cookie

    def decide(self, ticket: str, csrf: str, cookie: str, choice: str) -> str:
        with self.lock:
            row = self.consents.get(sha(ticket))
            if (row is None or row.consumed or self.clock.now() >= row.expires
                or not hmac.compare_digest(row.csrf_digest, sha(csrf))
                or not hmac.compare_digest(row.cookie_digest, sha(cookie))
                or choice not in {"approve", "deny"}):
                raise ProtocolError("invalid_consent", 403)
            row.consumed = True
            params = row.params
            response = {"state": params["state"], "iss": self.issuer}
            if choice == "deny":
                self.counts["denied"] += 1
                response["error"] = "access_denied"
            else:
                self.counts["approved"] += 1
                code = self.secret()
                self.codes[sha(code)] = CodeRecord(
                    CLIENT_ID, params["redirect_uri"], params["resource"],
                    parse_scope(params["scope"]), params["code_challenge"],
                    self.clock.now() + self.code_ttl,
                )
                response["code"] = code
            return params["redirect_uri"] + "?" + urlencode(response)

    def exchange(self, fields: dict) -> dict:
        # 2.1 草案主路径不发送 redirect_uri。若旧客户端发送，本课额外严格核对它。
        required = {"grant_type", "client_id", "code", "code_verifier", "resource"}
        with self.lock:
            self.counts["token_requests"] += 1
            if not required <= fields.keys() or fields.keys() - required - {"redirect_uri"}:
                raise ProtocolError("invalid_request")
            if fields["grant_type"] != "authorization_code":
                raise ProtocolError("unsupported_grant_type")
            record = self.codes.get(sha(fields["code"]))
            if record is None or self.clock.now() >= record.expires:
                raise ProtocolError("invalid_grant")
            if fields["client_id"] != record.client_id:
                raise ProtocolError("invalid_grant")
            if fields["resource"] != record.resource:
                raise ProtocolError("invalid_target")
            if "redirect_uri" in fields and fields["redirect_uri"] != record.redirect_uri:
                raise ProtocolError("invalid_grant")
            try:
                challenge = pkce_s256(fields["code_verifier"])
            except LessonError:
                raise ProtocolError("invalid_grant") from None
            if not hmac.compare_digest(challenge, record.challenge):
                raise ProtocolError("invalid_grant")
            if record.used:
                # 本课实现授权码重放时撤销此前由该码签发的 token。
                for key in record.token_digests:
                    self.tokens[key].revoked = True
                raise ProtocolError("invalid_grant")
            # 校验、消费和签发都在同一临界区；锁只保护本进程，不是分布式事务。
            record.used = True
            access_token = self.secret()
            digest = sha(access_token)
            self.tokens[digest] = TokenRecord("alice", self.issuer, record.resource,
                                               record.scopes, self.clock.now() + self.token_ttl)
            record.token_digests.append(digest)
            self.counts["tokens_issued"] += 1
            return {"access_token": access_token, "token_type": "Bearer",
                    "expires_in": self.token_ttl, "scope": " ".join(sorted(record.scopes))}

    def validate_access(self, token: str, audience: str, scope: str | None = None) -> TokenRecord:
        with self.lock:
            record = self.tokens.get(sha(token))
            if (record is None or record.revoked or self.clock.now() >= record.expires
                or record.issuer != self.issuer or record.audience != audience):
                raise ProtocolError("invalid_token", 401)
            if scope is not None and scope not in record.scopes:
                raise ProtocolError("insufficient_scope", 403)
            return copy.deepcopy(record)

## 5．客户端：先验证 metadata，再去授权

发现链是：

```text
访问受保护资源 → 401
    → 按固定 resource metadata 地址读取
    → 验证目标资源和允许的授权服务器
    → 读取已选 issuer 的 metadata
    → 检查 issuer、授权端点、token 端点、S256 和 iss 能力
```

不能先访问任意 metadata 给出的 URL，然后才想起验证目的地。本课更保守：只访问已经启动的三个回环地址，端点还必须使用固定路径；不是所有正规 OAuth 部署都要求同源固定路径。[R4][R5][R8]

回调 `iss` 与开始时选定的 issuer 做精确比较，**不能收到哪个 issuer 就信哪个 issuer**。[R6]

In [5]:
# %% 03 客户端的单次授权状态：state、issuer 和 PKCE verifier
@dataclass
class PendingFlow:
    issuer: str
    resource: str
    redirect_uri: str
    scopes: frozenset[str]
    state: str
    verifier: str
    expires: float
    status: str = "pending"
    code: str | None = None
    error: str | None = None
    event: threading.Event = field(default_factory=threading.Event, repr=False)
    lock: threading.Lock = field(default_factory=threading.Lock, repr=False)


def check_callback(params: dict, flow: PendingFlow, now: float) -> tuple[str, str]:
    """纯校验，不消费 flow。error 回调也必须验证 state 和 iss。"""
    require(flow.status == "pending" and now < flow.expires, "回调已经使用或到期")
    require(set(params) in ({"code", "state", "iss"}, {"error", "state", "iss"}),
            "回调字段错误")
    require(hmac.compare_digest(params.get("state", ""), flow.state), "state 不匹配")
    require(params.get("iss") == flow.issuer, "issuer 不匹配")
    if "error" in params:
        require(params["error"] == "access_denied", "未支持的授权错误")
        return "denied", "access_denied"
    require(bool(re.fullmatch(r"[A-Za-z0-9_-]{32,128}", params["code"])), "授权码格式错误")
    return "authorized", params["code"]


def accept_callback(params: dict, flow: PendingFlow, now: float) -> None:
    with flow.lock:
        status, value = check_callback(params, flow, now)
        flow.status = status
        if status == "authorized":
            flow.code = value
        else:
            flow.error = value
        flow.event.set()


def validate_discovery(resource_meta: dict, as_meta: dict,
                       expected_resource: str, pinned_issuer: str) -> None:
    require(resource_meta.get("resource") == expected_resource, "resource 不匹配")
    require(resource_meta.get("authorization_servers") == [pinned_issuer], "不可信发行方")
    require(as_meta.get("issuer") == pinned_issuer, "metadata issuer 不匹配")
    # 本课更保守：端点固定同源路径，不是 OAuth 对所有部署的通用限制。
    require(as_meta.get("authorization_endpoint") == pinned_issuer + "/authorize", "授权端点变了")
    require(as_meta.get("token_endpoint") == pinned_issuer + "/token", "令牌端点变了")
    require("S256" in as_meta.get("code_challenge_methods_supported", []), "缺少 S256")
    require(as_meta.get("authorization_response_iss_parameter_supported") is True, "缺少 issuer 响应能力")
    require(as_meta.get("token_endpoint_auth_methods_supported") == ["none"], "不是本课公共客户端")

### 不消耗正确请求的坏回调

本课在收到错误 state/issuer 时拒绝本次请求，但保留正确的待处理状态；合法回调成功后才原子消费。错误和过期的回调都不能触发 `/token`。

下面先用纯函数观察，不发送网络请求。

In [6]:
toy = PendingFlow("http://127.0.0.1:9001", "http://127.0.0.1:9002/mcp",
                  "http://127.0.0.1:9003/callback", frozenset({"docs:read"}),
                  "s" * 43, "v" * 43, 100)
good = {"state": toy.state, "iss": toy.issuer, "code": "c" * 43}
print("正确回调：", check_callback(good, toy, 10)[0])
for label, altered in [("错误state", {**good, "state": "wrong"}),
                       ("错误issuer", {**good, "iss": "wrong"})]:
    try:
        check_callback(altered, toy, 10)
    except LessonError as e:
        print(label, "→ 拒绝：", str(e))
print("纯校验后状态：", toy.status)

正确回调： authorized
错误state → 拒绝： state 不匹配
错误issuer → 拒绝： issuer 不匹配
纯校验后状态： pending


## 6．真实 HTTP，但不自动跟随任意重定向

`Wire` 只访问宿主事先登记的回环 origin，读取长度有上限，不自动跟随 3xx，也不把 Bearer 头传给授权页、metadata 或回调。

只有本课浏览器夹具在确认精确回调地址以后，才主动访问那个回调。实际交互模式则由用户浏览器完成这一跳。

审计仅保存 HTTP 方法、固定路径和状态，不保存 query、Location、Cookie、Authorization、token 响应或换码表单。浏览器地址栏仍可能短暂包含 code；页面不含外部资源，并设置 no-store 与 no-referrer。仍然不等于生产日志防泄漏体系。

In [7]:
# %% 04 有界 HTTP 客户端与审计：日志不保存凭证、授权码或 verifier
@dataclass
class WireResponse:
    status: int
    headers: dict[str, str]
    body: bytes = field(repr=False)
    def json(self):
        return strict_json(self.body.decode("utf-8"))

class Wire:
    def __init__(self):
        self.events: list[dict] = []
        self.allowed_origins: set[str] = set()
        self.lock = threading.Lock()

    def request(self, method: str, url: str, *, form=None, payload=None,
                headers: dict | None = None) -> WireResponse:
        p = loopback(url)
        origin = f"http://127.0.0.1:{p.port}"
        require(origin in self.allowed_origins, "未配置的网络目的地")
        require(form is None or payload is None, "不能同时发两种正文")
        body = None
        h = dict(headers or {})
        if form is not None:
            body = urlencode(form).encode("utf-8")
            h["Content-Type"] = "application/x-www-form-urlencoded"
        if payload is not None:
            body = canonical(payload).encode("utf-8")
            h["Content-Type"] = "application/json"
        h.setdefault("Accept", "application/json")
        conn = http.client.HTTPConnection("127.0.0.1", p.port, timeout=4)
        try:
            conn.request(method, p.path + ("?" + p.query if p.query else ""), body=body, headers=h)
            response = conn.getresponse()
            data = response.read(MAX_BODY + 1)
            require(len(data) <= MAX_BODY, "响应过大")
            out = WireResponse(response.status, {k.lower(): v for k, v in response.getheaders()}, data)
        finally:
            conn.close()
        with self.lock:
            # 不记录 query、Location、Cookie、Authorization 或任何请求/响应正文。
            self.events.append({"method": method, "path": p.path, "status": out.status})
        return out

class FlowClient:
    def __init__(self, lab):
        self.lab, self.flow, self.access_token = lab, None, None
        self.metadata_checked = False

    def discover(self) -> dict:
        w, a = self.lab.wire, self.lab.authority
        probe = self.call("tools/list", token="")
        require(probe.status == 401, "初次资源请求应该要求授权")
        expected_url = self.lab.resource_origin + RESOURCE_META
        challenge = probe.headers.get("www-authenticate", "")
        require(f'resource_metadata="{expected_url}"' in challenge, "意外的 metadata 地址")
        rm = w.request("GET", expected_url).json()
        # 先验证授权服务器列表，再发起 metadata 网络请求。
        require(rm.get("resource") == a.resource and rm.get("authorization_servers") == [a.issuer],
                "发现结果不在宿主可信配置中")
        am = w.request("GET", a.issuer + AS_META).json()
        validate_discovery(rm, am, a.resource, a.issuer)
        self.metadata_checked = True
        return {"unauthorized_status": probe.status, "pkce_methods": am["code_challenge_methods_supported"],
                "issuer_pinned": True, "resource_pinned": True}

    def begin(self, scope="docs:read") -> str:
        require(self.metadata_checked, "先验证 metadata")
        require(self.flow is None or self.flow.status != "pending", "已有待处理授权")
        a = self.lab.authority
        self.flow = PendingFlow(a.issuer, a.resource, a.redirect_uri, parse_scope(scope),
                                a.secret(), new_verifier(), a.clock.now() + 300)
        a.secret_values.append(self.flow.verifier)
        p = {"response_type": "code", "client_id": CLIENT_ID, "redirect_uri": a.redirect_uri,
             "scope": " ".join(sorted(self.flow.scopes)), "state": self.flow.state,
             "resource": a.resource, "code_challenge": pkce_s256(self.flow.verifier),
             "code_challenge_method": "S256"}
        return a.issuer + "/authorize?" + urlencode(p)

    def token_fields(self) -> dict:
        f = self.flow
        require(f is not None and f.status == "authorized", "没有可交换的成功授权回调")
        return {"grant_type": "authorization_code", "client_id": CLIENT_ID,
                "code": f.code, "code_verifier": f.verifier, "resource": f.resource}

    def exchange(self) -> dict:
        fields = self.token_fields()
        self.flow.status = "exchanging"
        try:
            r = self.lab.wire.request("POST", self.flow.issuer + "/token", form=fields)
            if r.status != 200:
                self.flow.status = "failed"
                raise ProtocolError(r.json().get("error", "token_error"), r.status)
            result = r.json()
            require(result.get("token_type") == "Bearer", "未知 token 类型")
            require(parse_scope(result["scope"]) <= self.flow.scopes, "返回权限超出请求")
            require(isinstance(result["expires_in"], int) and result["expires_in"] > 0, "过期时间错误")
            self.access_token = result["access_token"]
            self.flow.status = "complete"
            return {"status": r.status, "token_type": result["token_type"],
                    "scopes": sorted(parse_scope(result["scope"])), "expires_in": result["expires_in"]}
        except (OSError, http.client.HTTPException):
            # token 请求不盲目重试：授权码可能已被消费，但响应丢了。
            self.flow.status = "exchange_unknown"
            raise
        finally:
            self.flow.code, self.flow.verifier = None, ""

    def call(self, method: str, *, name: str | None = None, arguments=None,
             token: str | None = None) -> WireResponse:
        p = {"_meta": {META + "protocolVersion": PROTOCOL, META + "clientCapabilities": {}}}
        h = {"MCP-Protocol-Version": PROTOCOL, "Mcp-Method": method}
        if name is not None:
            p.update(name=name, arguments=arguments or {})
            h["Mcp-Name"] = name
        selected = self.access_token if token is None else token
        if selected:
            h["Authorization"] = "Bearer " + selected
        return self.lab.wire.request("POST", self.lab.authority.resource,
                payload={"jsonrpc": "2.0", "id": "lesson53-call", "method": method, "params": p}, headers=h)

### Token 端点使用 POST，不是把 verifier 放进 URL

本课成功回调后发送的字段：

```text
grant_type=authorization_code
client_id=lesson53-public-client
code=<收到的授权码>
code_verifier=<本地保留的随机秘密>
resource=<最初请求的资源>
```

没有 client_secret；没有新的 scope；主路径没有 redirect_uri。新权限要发起新的授权，不允许换码时悄悄加上管理员 scope。[R1][R5]

换码请求与“安全幂等的工具查询”不同。code 可能已经消费而响应丢失，本课不会自动重复换码。授权码重放被拒绝，且本课会撤销此前由该码签发的 token，因而不能盲目套用通用 HTTP 重试器。

## 7．三个 HTTP 入口的实现

| 角色 | 本课入口 |
|---|---|
| AS | metadata、GET `/authorize`、POST `/consent`、POST `/token` |
| 客户端 | GET `/callback` |
| RS | resource metadata、POST `/mcp` |

默认绑定 `127.0.0.1` 随机端口。宿主启动时显式登记本次精确回调，不提供开放注册接口。这是比原生客户端端口例外更窄的教学策略；RFC 8252 的正式原生 loopback 登记可以允许端口变化，不能据此推导任意路径也可变。[R7]

**正式 AS、token 端点和 RS 应使用 HTTPS。** 原生客户端 loopback callback 的 HTTP 例外，不是让授权服务器和资源 API 使用明文 HTTP 的通用许可。本课为完全本地运行而有意简化，不能宣称端到端协议合规。

MCP 部分只保留固定 `2026-07-28` 的 JSON 工具子集；不新增 SSE、完整工具schema能力或协议认证。

In [8]:
# %% 05 三个 HTTP 角色；公开身份固定 Alice，不模拟生产登录
class LabHTTPServer(ThreadingHTTPServer):
    daemon_threads = True
    allow_reuse_address = True

class LabHandler(BaseHTTPRequestHandler):
    protocol_version = "HTTP/1.0"
    def log_message(self, *args):
        pass  # 默认 HTTP 日志会带回调 query；这里不输出。

    def setup(self):
        super().setup()
        self.connection.settimeout(4)

    def respond(self, status: int, value=None, *, headers=None, text: str | None = None):
        body = (canonical(value).encode("utf-8") if text is None else text.encode("utf-8"))
        self.send_response(status)
        self.send_header("Content-Type", "application/json; charset=utf-8" if text is None else "text/html; charset=utf-8")
        self.send_header("Content-Length", str(len(body)))
        self.send_header("Cache-Control", "no-store")
        self.send_header("Pragma", "no-cache")
        self.send_header("Referrer-Policy", "no-referrer")
        self.send_header("X-Content-Type-Options", "nosniff")
        self.send_header("X-Frame-Options", "DENY")
        self.send_header("Content-Security-Policy", "default-src 'none'; form-action 'self'; frame-ancestors 'none'")
        for k, v in (headers or {}).items():
            self.send_header(k, v)
        self.end_headers()
        self.wfile.write(body)

    def read_body(self, media: str) -> bytes:
        lengths = self.headers.get_all("Content-Length", [])
        require(len(lengths) == 1 and lengths[0].isdigit(), "缺少或重复长度")
        length = int(lengths[0])
        require(0 < length <= MAX_BODY, "请求体长度错误")
        require(self.headers.get("Transfer-Encoding") is None, "本课不支持传输编码")
        require(self.headers.get_content_type() == media, "媒体类型错误")
        body = self.rfile.read(length)
        require(len(body) == length, "不完整请求")
        return body

    def do_GET(self):
        self.handle_request("GET")
    def do_POST(self):
        self.handle_request("POST")

    def handle_request(self, verb: str):
        lab, role = self.server.lab, self.server.role
        try:
            expected_host = f"127.0.0.1:{self.server.server_port}"
            require(self.headers.get_all("Host", []) == [expected_host], "Host 不匹配")
            require(len(self.path) <= MAX_BODY, "URL 过长")
            path = urlsplit(self.path)
            if role == "authorization":
                return self.authorization(lab, verb, path)
            if role == "resource":
                return self.resource(lab, verb, path)
            if role == "callback":
                require(verb == "GET" and path.path == "/callback", "回调路由错误")
                require(lab.client.flow is not None, "没有待处理授权")
                accept_callback(strict_form(path.query), lab.client.flow, lab.authority.clock.now())
                # 不在浏览器响应中显示授权码或 access token。
                return self.respond(200, text="<meta charset='utf-8'><h1>回调已接收</h1><p>可关闭此页面，回到终端。</p>")
            raise LessonError("角色错误")
        except ProtocolError as e:
            headers = {}
            if role == "resource" and e.status == 401:
                headers["WWW-Authenticate"] = 'Bearer error="invalid_token", resource_metadata="' + lab.resource_origin + RESOURCE_META + '"'
            if role == "resource" and e.status == 403:
                headers["WWW-Authenticate"] = 'Bearer error="insufficient_scope"'
            self.respond(e.status, {"error": e.error}, headers=headers)
        except (LessonError, ValueError, KeyError, TypeError):
            self.respond(400, {"error": "invalid_request"})
        except (BrokenPipeError, ConnectionResetError, TimeoutError):
            return

    def authorization(self, lab, verb, path):
        a = lab.authority
        if verb == "GET" and path.path == AS_META and not path.query:
            return self.respond(200, a.metadata())
        if verb == "GET" and path.path == "/authorize":
            p = strict_form(path.query)
            ticket, csrf, cookie = a.begin_consent(p)
            escaped = {k: html.escape(v, quote=True) for k, v in p.items()}
            page = f"""<!doctype html><html lang="zh"><meta charset="utf-8">
<title>第53课：本地模拟授权</title><h1>模拟 Alice 的同意页</h1>
<p>本页没有验证用户身份，不是真实账号登录。只使用本地教学工具。</p>
<p>客户端：{escaped['client_id']}</p><p>目标资源：{escaped['resource']}</p>
<p>请求权限：{escaped['scope']}</p><p>回调：{escaped['redirect_uri']}</p>
<form action="/consent" method="post">
<input type="hidden" name="ticket" value="{ticket}">
<input type="hidden" name="csrf" value="{csrf}">
<button name="decision" value="approve" type="submit">批准这些权限</button>
<button name="decision" value="deny" type="submit">拒绝</button></form></html>"""
            return self.respond(200, text=page, headers={
                "Set-Cookie": f"{a.cookie_name}={cookie}; HttpOnly; SameSite=Strict; Path=/consent"})
        if verb == "POST" and path.path == "/consent" and not path.query:
            require(self.headers.get_all("Origin", []) == [a.issuer], "审批 Origin 错误")
            form = strict_form(self.read_body("application/x-www-form-urlencoded").decode("utf-8"))
            require(set(form) == {"ticket", "csrf", "decision"}, "审批字段错误")
            raw = self.headers.get_all("Cookie", [])
            require(len(raw) == 1, "缺少审批 Cookie")
            cookie_pairs = [x.strip().split("=", 1) for x in raw[0].split(";") if "=" in x]
            vals = [v for k, v in cookie_pairs if k == a.cookie_name]
            require(len(vals) == 1, "审批 Cookie 错误")
            location = a.decide(form["ticket"], form["csrf"], vals[0], form["decision"])
            return self.respond(303, {"status": "redirect"}, headers={"Location": location})
        if verb == "POST" and path.path == "/token" and not path.query:
            require(not self.headers.get_all("Authorization"), "公共客户端不发送 client_secret")
            form = strict_form(self.read_body("application/x-www-form-urlencoded").decode("utf-8"))
            return self.respond(200, a.exchange(form))
        return self.respond(404, {"error": "not_found"})

    def resource(self, lab, verb, path):
        a = lab.authority
        if verb == "GET" and path.path == RESOURCE_META and not path.query:
            return self.respond(200, {"resource": a.resource, "authorization_servers": [a.issuer],
                                      "scopes_supported": sorted(SCOPES)})
        require(path.path == "/mcp" and not path.query, "资源路由错误；不接受 URL token")
        if verb != "POST":
            return self.respond(405, {"error": "method_not_allowed"}, headers={"Allow": "POST"})
        raw_auth = self.headers.get_all("Authorization", [])
        if len(raw_auth) != 1 or not raw_auth[0].startswith("Bearer "):
            raise ProtocolError("invalid_token", 401)
        token = raw_auth[0][7:]
        identity = a.validate_access(token, a.resource)
        request = strict_json(self.read_body("application/json").decode("utf-8"))
        require(isinstance(request, dict) and set(request) == {"jsonrpc", "id", "method", "params"}, "RPC 外壳错误")
        require(request["jsonrpc"] == "2.0" and isinstance(request["id"], str), "RPC id 错误")
        p, method = request["params"], request["method"]
        require(isinstance(p, dict), "参数必须是字典")
        require(p.get("_meta", {}).get(META + "protocolVersion") == PROTOCOL, "协议版本错误")
        require(p.get("_meta", {}).get(META + "clientCapabilities") == {}, "本课只支持空客户端能力")
        require(self.headers.get_all("MCP-Protocol-Version", []) == [PROTOCOL], "版本镜像错误")
        require(self.headers.get_all("Mcp-Method", []) == [method], "方法镜像错误")
        def done(result):
            return self.respond(200, {"jsonrpc": "2.0", "id": request["id"],
                                       "result": {"resultType": "complete", **result}})
        if method == "server/discover":
            return done({"serverInfo": {"name": "lesson53-tools", "version": "1"},
                         "supportedVersions": [PROTOCOL], "capabilities": {"tools": {}}})
        if method == "tools/list":
            catalog = []
            if "docs:read" in identity.scopes:
                catalog.append({"name": "read_doc", "description": "读取固定教学文档",
                    "inputSchema": {"type": "object", "properties": {"doc_id": {"type": "string"}},
                                    "required": ["doc_id"], "additionalProperties": False}})
            if "calc:use" in identity.scopes:
                catalog.append({"name": "calculate", "description": "有界整数减法",
                    "inputSchema": {"type": "object", "properties": {k: {"type": "integer"} for k in ["left", "right"]},
                                    "required": ["left", "right"], "additionalProperties": False}})
            return done({"tools": catalog})
        require(method == "tools/call" and set(p) == {"_meta", "name", "arguments"}, "不支持的工具请求")
        name, arguments = p["name"], p["arguments"]
        require(self.headers.get_all("Mcp-Name", []) == [name], "工具镜像错误")
        if name not in {"read_doc", "calculate"}:
            return self.respond(200, {"jsonrpc": "2.0", "id": request["id"],
                                     "error": {"code": -32602, "message": "unknown tool"}})
        a.validate_access(token, a.resource, "docs:read" if name == "read_doc" else "calc:use")
        if name == "read_doc":
            require(isinstance(arguments, dict) and set(arguments) == {"doc_id"}, "文档参数错误")
            if arguments["doc_id"] == "bob_private":
                raise ProtocolError("insufficient_scope", 403)  # 对象 ACL，scope 不是全部对象的通行证。
            docs = {"backup": "自动备份保留14天。", "recycle": "回收站保留30天。"}
            if arguments["doc_id"] not in docs:
                return done({"isError": True, "content": [{"type": "text", "text": "not_found"}]})
            data = {"text": docs[arguments["doc_id"]], "subject": identity.subject}
        else:
            require(isinstance(arguments, dict) and set(arguments) == {"left", "right"}, "计算参数错误")
            require(all(type(v) is int and abs(v) <= 1000000 for v in arguments.values()), "整数范围错误")
            data = {"value": arguments["left"] - arguments["right"]}
        with a.lock:
            a.counts["resource_executions"] += 1
        return done({"isError": False, "structuredContent": data,
                     "content": [{"type": "text", "text": canonical(data)}]})

In [9]:
# %% 06 本地实验装配；浏览器自动化只操作本课页面
class HiddenInputs(HTMLParser):
    def __init__(self):
        super().__init__()
        self.values = {}
    def handle_starttag(self, tag, attrs):
        a = dict(attrs)
        if tag == "input" and a.get("type") == "hidden":
            require(a["name"] not in self.values, "页面隐藏字段重复")
            self.values[a["name"]] = a.get("value", "")

class LocalLab(AbstractContextManager):
    def __init__(self):
        self.authority, self.wire = Authority(), Wire()
        self.servers, self.threads = [], []
        self.client = FlowClient(self)

    def __enter__(self):
        try:
            origins = {}
            for role in ("authorization", "resource", "callback"):
                server = LabHTTPServer(("127.0.0.1", 0), LabHandler)
                server.lab, server.role = self, role
                self.servers.append(server)
                origin = f"http://127.0.0.1:{server.server_port}"
                origins[role] = origin
                self.wire.allowed_origins.add(origin)
            a = self.authority
            a.issuer, self.resource_origin = origins["authorization"], origins["resource"]
            a.resource = self.resource_origin + "/mcp"
            a.redirect_uri = origins["callback"] + "/callback"
            # 本课启动时显式登记本次精确回调，不实现动态注册或原生端口通配规范。
            for server in self.servers:
                thread = threading.Thread(target=server.serve_forever,
                    kwargs={"poll_interval": 0.02}, daemon=True)
                thread.start()
                self.threads.append(thread)
            return self
        except BaseException:
            self.__exit__(*sys.exc_info())
            raise

    def __exit__(self, *args):
        for server, thread in zip(self.servers, self.threads):
            if thread.is_alive():
                server.shutdown()
        for server in self.servers:
            server.server_close()
        for thread in self.threads:
            thread.join(timeout=3)
        return False

    def browser_decision(self, authorization_url: str, choice="approve") -> str:
        """测试夹具：模拟浏览器同源表单往返。不是人工审阅成绩。"""
        w, a = self.wire, self.authority
        require(authorization_url.startswith(a.issuer + "/authorize?"), "不是本课授权地址")
        response = w.request("GET", authorization_url)
        require(response.status == 200, "授权页面返回错误")
        fields = HiddenInputs()
        fields.feed(response.body.decode("utf-8"))
        cookie = response.headers["set-cookie"].split(";", 1)[0]
        result = w.request("POST", a.issuer + "/consent",
                 form={**fields.values, "decision": choice},
                 headers={"Cookie": cookie, "Origin": a.issuer})
        require(result.status == 303, "审批没有返回回调")
        location = result.headers["location"]
        p = urlsplit(location)
        require(location.split("?", 1)[0] == a.redirect_uri and not p.fragment, "回调目的地不匹配")
        return location  # 主动返回，不会跟随任意重定向，也不携带 Bearer。

    def callback(self, location: str) -> WireResponse:
        require(location.split("?", 1)[0] == self.authority.redirect_uri, "不能回调未知客户端")
        return self.wire.request("GET", location)

    def authorize(self, scopes="docs:read", choice="approve") -> dict:
        if not self.client.metadata_checked:
            self.client.discover()
        url = self.client.begin(scopes)
        location = self.browser_decision(url, choice)
        response = self.callback(location)
        require(response.status == 200, "客户端回调被拒绝")
        if self.client.flow.status == "denied":
            return {"status": "denied", "error": "access_denied", "token_request_sent": False}
        return self.client.exchange()

## 8．第一次完整授权：只读 token 无法计算

实际启动三个监听器，经历发现、同意表单、303 回调、换码与受保护工具调用。默认批准由 `browser_decision()` 模拟，不是真人决定。

资源先检查 token 的有效性与 audience，再检查工具 scope；对象 ACL 也仍然存在。`docs:read` 不能读 Bob 的私人资料。目录隐藏工具仅为便利，调用时仍然重新验证。

In [10]:
with LocalLab() as lab:
    print("发现：", lab.client.discover())
    print("批准只读：", lab.authorize("docs:read"))
    read = lab.client.call("tools/call", name="read_doc", arguments={"doc_id": "backup"})
    before = lab.authority.counts["resource_executions"]
    calc = lab.client.call("tools/call", name="calculate", arguments={"left": 30, "right": 14})
    print("读取状态：", read.status, "正文：", read.json()["result"]["structuredContent"]["text"])
    print("计算状态：", calc.status)
    print("拒绝计算没有执行：", before == lab.authority.counts["resource_executions"])
print("上下文结束，三个监听器已关闭。")

发现： {'unauthorized_status': 401, 'pkce_methods': ['S256'], 'issuer_pinned': True, 'resource_pinned': True}
批准只读： {'status': 200, 'token_type': 'Bearer', 'scopes': ['docs:read'], 'expires_in': 120}
读取状态： 200 正文： 自动备份保留14天。
计算状态： 403
拒绝计算没有执行： True
上下文结束，三个监听器已关闭。


## 9．扩权限需要新的授权，旧 token 不会自动升级

本课同一个客户端先申请 `docs:read`，随后重新申请 `docs:read calc:use` 并再次批准。

新 token 可以计算，但旧 token 仍然只读。scope 表达操作范围，resource/audience 表达给哪个服务用；两项缺一不可。[R5][R8]

用户对读取工具的授权也不等于第45课的“批准某份代码执行”。OAuth 授权、宿主白名单、具体写操作审批仍然是不同层次。

In [11]:
with LocalLab() as lab:
    lab.authorize("docs:read")
    old_token = lab.client.access_token
    print("新的授权：", lab.authorize("docs:read calc:use"))
    good = lab.client.call("tools/call", name="calculate", arguments={"left": 30, "right": 14})
    old = lab.client.call("tools/call", name="calculate", arguments={"left": 30, "right": 14}, token=old_token)
    print("新token：", good.status, good.json()["result"]["structuredContent"])
    print("旧token仍被拒绝：", old.status)
    del old_token

新的授权： {'status': 200, 'token_type': 'Bearer', 'scopes': ['calc:use', 'docs:read'], 'expires_in': 120}
新token： 200 {'value': 16}
旧token仍被拒绝： 403


## 10．拒绝授权是完整结果，不是异常后偷偷默认同意

用户点击拒绝，AS 回调 `error=access_denied`，客户端核对 state/iss 后结束。它不应该继续交换 token、把旧凭证解释为本次新批准，也不应该无限重新弹授权页。

本课不会在拒绝时清除一个独立的旧 grant，但这不赋予它新权限。下面用全新环境演示，拒绝后没有任何 token 请求。

In [12]:
with LocalLab() as lab:
    denied = lab.authorize("docs:read", choice="deny")
    print(denied)
    print("AS收到token请求次数：", lab.authority.counts["token_requests"])
    print("签发token数量：", lab.authority.counts["tokens_issued"])

{'status': 'denied', 'error': 'access_denied', 'token_request_sent': False}
AS收到token请求次数： 0
签发token数量： 0


## 11．code、verifier、access token 分别能做什么？

接下来使用自己的本地授权数据，复现：拿到 code、但使用错误 verifier 换码失败；正确 verifier 可以成功；code 不能作为资源 Bearer；成功换码后重放 code 被拒绝。

**PKCE 不保护已经泄漏的普通 Bearer token。** 本课没有 DPoP、mTLS 或 sender-constrained token，谁拿到有效 Bearer，通常就能在其 scope 与 audience 范围内使用它。不要把 PKCE 宣称为万能令牌防盗技术。[R1][R3]

故障只针对本课创建的服务与数据，不接触真实用户账号或第三方服务。

In [13]:
# %% 07 完整实验和负例：结果与秘密分开

def run_demo() -> dict:
    with LocalLab() as lab:
        c, a = lab.client, lab.authority
        discovery = c.discover()
        reader = lab.authorize("docs:read")
        reader_token = c.access_token  # 仅内存，不进入报告。
        read = c.call("tools/call", name="read_doc", arguments={"doc_id": "backup"})
        before = a.counts["resource_executions"]
        denied_calc = c.call("tools/call", name="calculate", arguments={"left": 30, "right": 14})
        blocked_before_execution = before == a.counts["resource_executions"]
        old_scopes = a.validate_access(reader_token, a.resource).scopes
        escalation = lab.authorize("docs:read calc:use")
        calc = c.call("tools/call", name="calculate", arguments={"left": 30, "right": 14})
        old_token_calc = c.call("tools/call", name="calculate", arguments={"left": 30, "right": 14}, token=reader_token)
        token_count = a.counts["token_requests"]
        denial = lab.authorize("docs:read", choice="deny")
        no_token_request_after_deny = a.counts["token_requests"] == token_count
        c.begin("docs:read")
        # 只掌握授权码但没有正确 verifier：用本课自己的测试数据复现，不触及外部服务。
        auth_url = a.issuer + "/authorize?" + urlencode({
            "response_type": "code", "client_id": CLIENT_ID, "redirect_uri": a.redirect_uri,
            "scope": "docs:read", "state": c.flow.state, "resource": a.resource,
            "code_challenge": pkce_s256(c.flow.verifier), "code_challenge_method": "S256"})
        location = lab.browser_decision(auth_url)
        lab.callback(location)
        fields = c.token_fields()
        wrong = lab.wire.request("POST", a.issuer + "/token", form={**fields, "code_verifier": "Z" * 43})
        code_as_bearer = c.call("tools/list", token=fields["code"])
        issued = c.exchange()
        valid_token = c.access_token
        replay = lab.wire.request("POST", a.issuer + "/token", form=fields)
        replay_token = c.call("tools/list", token=valid_token)
        report = {
            "mode": "actual_loopback_http_scripted_consent", "oauth21": "draft-17; not an RFC",
            "discovery": discovery, "reader_grant": reader,
            "reader_read_status": read.status,
            "reader_calculate_status": denied_calc.status,
            "scope_denial_did_not_execute": blocked_before_execution,
            "second_grant": escalation,
            "calculate_value": calc.json()["result"]["structuredContent"]["value"],
            "old_token_scopes": sorted(old_scopes), "old_token_calculate_status": old_token_calc.status,
            "denial": denial, "deny_did_not_exchange": no_token_request_after_deny,
            "wrong_verifier_status": wrong.status, "code_used_as_bearer_status": code_as_bearer.status,
            "correct_verifier_status": issued["status"], "code_replay_status": replay.status,
            "token_after_code_replay_status": replay_token.status,
            "replay_policy": "本课在有效原请求重放授权码时拒绝并撤销原码签发的token",
            "counts": dict(a.counts), "wire": list(lab.wire.events),
            "limits": ["Alice 是固定教学身份，没有验证真实用户",
                       "AS与RS只有逻辑和端口隔离，可信token验证器共享内存",
                       "全链路HTTP仅供回环教学；正式AS/RS需HTTPS",
                       "无refresh/JWT/DPoP/真实身份系统/持久令牌库/生产限流",
                       "三个线程监听器，不是三台远端服务，也没有模型与GPU"],
        }
        serialized = canonical(report)
        require(not any(value in serialized for value in a.secret_values), "报告泄漏秘密")
        return report


def run_interactive(timeout_seconds=180) -> dict:
    """在用户本机开启三个监听器；只显示授权页面 URL，不显示 code/token/verifier。"""
    with LocalLab() as lab:
        lab.client.discover()
        url = lab.client.begin("docs:read")
        print("这是固定 Alice 的本地模拟同意页，不是真实账号登录。")
        print("请在同一台电脑浏览器打开以下地址，批准或拒绝；完成后返回终端：")
        print(url)
        # 不强制打开浏览器，远程Notebook/无GUI时由用户自行使用同机终端。
        if not lab.client.flow.event.wait(timeout_seconds):
            lab.client.flow.status = "expired"
            return {"mode": "interactive", "status": "local_wait_expired"}
        if lab.client.flow.status == "denied":
            return {"mode": "interactive", "status": "denied", "token_requests": 0}
        grant = lab.client.exchange()
        read = lab.client.call("tools/call", name="read_doc", arguments={"doc_id": "backup"})
        calc = lab.client.call("tools/call", name="calculate", arguments={"left": 30, "right": 14})
        return {"mode": "interactive", "grant": grant, "read_status": read.status,
                "read_data": read.json()["result"]["structuredContent"],
                "calculate_status": calc.status, "wire": lab.wire.events,
                "identity_is_simulated": True}

In [14]:
experiment = run_demo()
print("错误verifier换码：", experiment["wrong_verifier_status"])
print("正确verifier换码：", experiment["correct_verifier_status"])
print("把code当Bearer：", experiment["code_used_as_bearer_status"])
print("成功换码后重放code：", experiment["code_replay_status"])
print("重放后原token状态：", experiment["token_after_code_replay_status"])
print("策略说明：", experiment["replay_policy"])

错误verifier换码： 400
正确verifier换码： 200
把code当Bearer： 401
成功换码后重放code： 400
重放后原token状态： 401
策略说明： 本课在有效原请求重放授权码时拒绝并撤销原码签发的token


### 到期与原子消费

本课 code 有效期为60秒，access token 为120秒，同意页和客户端待处理授权最多300秒。检查使用 `now >= expires` 作为失效边界，测试通过推进受控时钟验证，不睡眠几分钟。

AS 的检查、消费和签发在同一把锁中完成。两条并发换码请求只会有一个得到200，另一条得到400；本课的重放撤销策略还可能让第一条新token失效。

这是本进程内的互斥，不是多实例数据库事务。进程重启会失去所有内存状态；没有持久令牌库、分布式锁、恢复或完整暴力重试限流。

## 12．交互模式：在自己的浏览器批准或拒绝

终端运行：

```bash
python lesson_53_oauth_authorization_code_and_pkce.py --interactive
```

程序打印一个本地授权页面 URL。使用**同一台电脑的浏览器**打开，检查页面中的客户端、资源与scope，再选择批准或拒绝。不要填写真实密码；页面只是模拟 Alice。

本次申请只读权限。批准后终端应显示：读取200、计算403；拒绝时不会换取token。脚本在流程结束或本地等待超时后关闭监听器。默认等待180秒，可通过 `--wait-seconds` 设置10–600秒。

不要在云端或远程Notebook里把服务器的127.0.0.1误认为你电脑的127.0.0.1。此练习应在本机终端运行，也不要把监听地址改成0.0.0.0来迁就访问。

授权URL包含短期 state/challenge。原始 code/token/verifier 不打印、不写入报告；页面 URL 也不适合贴进公开记录。

In [15]:
RUN_INTERACTIVE = False  # 只有本机交互环境才开启；默认全部执行不会等待浏览器。
if RUN_INTERACTIVE:
    interactive_result = run_interactive(timeout_seconds=180)
    print(canonical({k:v for k,v in interactive_result.items() if k != "wire"}))
else:
    print("跳过浏览器交互；离线HTTP实验已经使用脚本化同意。")

跳过浏览器交互；离线HTTP实验已经使用脚本化同意。


## 13．独立练习

先写自己的实现，再打开相应检查开关。练习函数默认抛出 `NotImplementedError`，但不自动调用，因此不会把空实现伪装成完成。

A 完成 S256；B 是纯回调校验，不能直接修改 flow；C 接收的是**服务端可信记录**，不是客户端自己声称的 JSON token。C 要区分401与403，并且不能修改原记录。

In [16]:
# %% 10 独立练习的检查入口：学生实现不被参考实现覆盖

def check_exercise_a(fn):
    require(fn(RFC_VERIFIER) == RFC_CHALLENGE, "S256 结果不符")
    require(len(fn("a"*128)) == 43 and "=" not in fn("a"*43), "边界或填充错误")
    for bad in ["abc", "中"*43, "a"*129]:
        try: fn(bad)
        except (ValueError, LessonError): pass
        else: raise AssertionError("必须拒绝非法verifier")
    return "练习A检查通过"


def check_exercise_b(fn):
    f = PendingFlow("http://127.0.0.1:8001", "http://127.0.0.1:8002/mcp", "http://127.0.0.1:8003/callback",
                    frozenset({"docs:read"}), "s"*43, "v"*43, 100)
    good = {"state":f.state,"iss":f.issuer,"code":"c"*43}
    require(fn(good, f, 10) == ("authorized", "c"*43), "合法回调错误")
    require(fn({"state":f.state,"iss":f.issuer,"error":"access_denied"},f,10)==("denied","access_denied"), "拒绝流程错误")
    for data, now in [({**good,"state":"wrong"},10),({**good,"iss":"wrong"},10),(good,100),
                      ({**good,"error":"access_denied"},10)]:
        try: fn(data,f,now)
        except (ValueError, LessonError): pass
        else: raise AssertionError("必须拒绝错误或过期回调")
    require(f.status=="pending", "纯校验函数不应修改flow")
    return "练习B检查通过"


def authorize_record(record: dict, issuer: str, audience: str, scope: str, now: float) -> str:
    if (record.get("issuer") != issuer or record.get("audience") != audience
        or record.get("revoked") is not False or now >= record.get("expires", -1)):
        raise ProtocolError("invalid_token",401)
    if scope not in record.get("scopes", []):
        raise ProtocolError("insufficient_scope",403)
    return record["subject"]


def check_exercise_c(fn):
    record={"issuer":"AS","audience":"RS","subject":"alice","scopes":["docs:read"],"expires":100,"revoked":False}
    saved=copy.deepcopy(record)
    require(fn(record,"AS","RS","docs:read",1)=="alice", "合法凭证错误")
    for r, aud, s, now, expected in [(record,"OTHER","docs:read",1,401),
             (record,"RS","docs:read",100,401), ({**record,"revoked":True},"RS","docs:read",1,401),
             (record,"RS","calc:use",1,403)]:
        try: fn(r,"AS",aud,s,now)
        except ProtocolError as e: require(e.status==expected,"401/403未区分")
        else: raise AssertionError("非法访问未拒绝")
    require(record==saved,"不能修改原记录")
    return "练习C检查通过"

### 练习 A：正确生成 S256 challenge

实现 `exercise_pkce(verifier)`。允许长度43–128且只含规范ASCII字符；计算摘要的原始字节，URL-safe Base64并去掉填充。非法输入抛 `LessonError` 或 `ValueError`。

In [17]:
def exercise_pkce(verifier):
    raise NotImplementedError("请实现S256")

RUN_EXERCISE_A = False
if RUN_EXERCISE_A:
    print(check_exercise_a(exercise_pkce))

### 练习 B：验证授权回调

实现 `exercise_callback(params, flow, now)`。成功返回 `("authorized", code)`，合法拒绝返回 `("denied", "access_denied")`。

检查待处理状态、有效期、字段结构、state与issuer；code/error不能同时出现。发生错误抛 `LessonError`，纯校验后不改变 flow。

In [18]:
def exercise_callback(params, flow, now):
    raise NotImplementedError("请校验回调")

RUN_EXERCISE_B = False
if RUN_EXERCISE_B:
    print(check_exercise_b(exercise_callback))

### 练习 C：资源授权

实现 `exercise_authorize(record, issuer, audience, scope, now)`。先校验issuer、audience、撤销和过期，再检查scope。凭证失效抛 `ProtocolError("invalid_token",401)`，权限不足抛 `ProtocolError("insufficient_scope",403)`，合法返回subject。

不要用显示名代替issuer，不要把scope检查放在实际工具执行之后。

In [19]:
def exercise_authorize(record, issuer, audience, scope, now):
    raise NotImplementedError("请实现资源侧权限检查")

RUN_EXERCISE_C = False
if RUN_EXERCISE_C:
    print(check_exercise_c(exercise_authorize))

## 14．参考实现：完成练习后再看

下方使用独立函数名，不覆盖学生实现。它们只解决本课约定的子集，不代替完整OAuth库。

In [20]:
def reference_pkce(verifier):
    if not isinstance(verifier, str) or VERIFIER_RE.fullmatch(verifier) is None:
        raise LessonError("verifier格式错误")
    digest = hashlib.sha256(verifier.encode("ascii")).digest()
    return base64.urlsafe_b64encode(digest).decode("ascii").rstrip("=")

print(check_exercise_a(reference_pkce))

练习A检查通过


In [21]:
def reference_callback(params, flow, now):
    if flow.status != "pending" or now >= flow.expires:
        raise LessonError("不是有效待处理授权")
    if set(params) not in ({"state", "iss", "code"}, {"state", "iss", "error"}):
        raise LessonError("回调字段不合法")
    if not hmac.compare_digest(params["state"], flow.state):
        raise LessonError("state串线")
    if params["iss"] != flow.issuer:
        raise LessonError("issuer串线")
    if "error" in params:
        if params["error"] != "access_denied":
            raise LessonError("未支持的拒绝原因")
        return "denied", "access_denied"
    if not re.fullmatch(r"[A-Za-z0-9_-]{32,128}", params["code"]):
        raise LessonError("授权码格式错误")
    return "authorized", params["code"]

print(check_exercise_b(reference_callback))

练习B检查通过


In [22]:
def reference_authorize(record, issuer, audience, scope, now):
    valid = (record.get("issuer") == issuer
             and record.get("audience") == audience
             and record.get("revoked") is False
             and now < record.get("expires", -1))
    if not valid:
        raise ProtocolError("invalid_token", 401)
    if scope not in record.get("scopes", []):
        raise ProtocolError("insufficient_scope", 403)
    return record["subject"]

print(check_exercise_c(reference_authorize))

练习C检查通过


## 15．完整自动检查与故障注入

检查包括 RFC7636 公共向量、非法verifier、错误回调、审批nonce、正确与错误换码、scope、对象ACL、到期边界、并发一次性消费和服务关闭。

每次独立HTTP检查在上下文结束时关闭监听器。错误示例均使用本课的随机凭证，没有真实账号。检查通过不是安全认证、渗透测试或跨平台兼容声明。

In [23]:
# %% 08 自动检查：既检查数学与状态，也实际发送 HTTP
class Checks:
    def __init__(self):
        self.names = []
    def eq(self, name, actual, expected):
        if actual != expected:
            raise AssertionError(f"{name}: actual={actual!r}, expected={expected!r}")
        self.names.append(name)
    def ok(self, name, value):
        self.eq(name, bool(value), True)
    def raises(self, name, callback, exc=LessonError):
        try:
            callback()
        except exc:
            self.names.append(name)
        else:
            raise AssertionError(name + ": 没有拒绝")
    def report(self):
        return {"count": len(self.names), "passed": self.names, "real_identity_login_tested": False}


def run_checks() -> dict:
    t = Checks()
    t.eq("RFC7636 S256 官方测试向量", pkce_s256(RFC_VERIFIER), RFC_CHALLENGE)
    t.eq("挑战无填充", "=" in pkce_s256(RFC_VERIFIER), False)
    t.eq("挑战43字符", len(pkce_s256(RFC_VERIFIER)), 43)
    for value in ["short", "a" * 129, "中" * 43, "a" * 42 + "!"]:
        t.raises("拒绝非法verifier-" + str(len(t.names)), lambda v=value: pkce_s256(v))
    t.eq("43边界合法", len(pkce_s256("a" * 43)), 43)
    t.eq("128边界合法", len(pkce_s256("a" * 128)), 43)
    t.ok("随机verifier满足格式", bool(VERIFIER_RE.fullmatch(new_verifier())))
    t.raises("重复表单键", lambda: strict_form("code=a&code=b"))
    t.raises("损坏百分号", lambda: strict_form("code=%zz"))
    t.raises("重复JSON键", lambda: strict_json('{"x":1,"x":2}'))
    t.raises("不访问外部地址", lambda: loopback("https://example.com/token"))
    t.raises("不允许URL用户信息", lambda: loopback("http://x@127.0.0.1:1234/token"))
    t.raises("未知scope", lambda: parse_scope("admin"))
    t.raises("重复scope", lambda: parse_scope("docs:read docs:read"))
    t.eq("scope顺序不改集合", parse_scope("calc:use docs:read"), SCOPES)
    with LocalLab() as lab:
        a, c, w = lab.authority, lab.client, lab.wire
        d = c.discover()
        t.eq("未授权资源触发401", d["unauthorized_status"], 401)
        t.eq("发布S256能力", d["pkce_methods"], ["S256"])
        am = a.metadata()
        rm = {"resource": a.resource, "authorization_servers": [a.issuer]}
        for field, value in [("issuer", "http://127.0.0.1:9"), ("token_endpoint", "http://127.0.0.1:9/token"),
                             ("code_challenge_methods_supported", ["plain"])]:
            t.raises("发现拒绝-" + field, lambda f=field, v=value: validate_discovery(rm, {**am, f:v}, a.resource, a.issuer))
        url = c.begin()
        params = strict_form(urlsplit(url).query)
        t.eq("授权请求不含verifier", "code_verifier" in params, False)
        t.eq("公开client不含secret", "client_secret" in params, False)
        for name, changed in [("回调变更", {"redirect_uri": a.redirect_uri + "/evil"}),
                              ("plain降级", {"code_challenge_method": "plain"}),
                              ("implicit模式", {"response_type": "token"}),
                              ("任意权限", {"scope": "admin"}),
                              ("另一个资源", {"resource": a.resource + "/other"})]:
            r = w.request("GET", a.issuer + "/authorize?" + urlencode({**params, **changed}))
            t.eq("授权入口拒绝-" + name, r.status, 400)
            t.eq("错误请求不重定向-" + name, "location" in r.headers, False)
        missing = dict(params); missing.pop("code_challenge")
        t.eq("缺少challenge拒绝", w.request("GET", a.issuer + "/authorize?" + urlencode(missing)).status, 400)
        # 真实表单：错误 CSRF 不会批准；随后仍可用正确表单。
        page = w.request("GET", url)
        hidden = HiddenInputs(); hidden.feed(page.body.decode())
        cookie = page.headers["set-cookie"].split(";", 1)[0]
        h = {"Cookie": cookie, "Origin": a.issuer}
        bad = w.request("POST", a.issuer + "/consent", form={**hidden.values, "csrf": "bad", "decision": "approve"}, headers=h)
        t.eq("同意表单错误CSRF拒绝", bad.status, 403)
        approved = w.request("POST", a.issuer + "/consent", form={**hidden.values, "decision": "approve"}, headers=h)
        t.eq("同意返回303", approved.status, 303)
        t.eq("一次性同意票据", w.request("POST", a.issuer + "/consent", form={**hidden.values, "decision": "approve"}, headers=h).status, 403)
        location = approved.headers["location"]
        q = strict_form(urlsplit(location).query)
        t.eq("授权回调不含access_token", "access_token" in q, False)
        for name, changed in [("state", {"state": "Z" * 43}), ("issuer", {"iss": a.issuer + "/wrong"})]:
            response = lab.callback(a.redirect_uri + "?" + urlencode({**q, **changed}))
            t.eq("回调拒绝-" + name, response.status, 400)
        t.eq("坏回调未触发token请求", a.counts["token_requests"], 0)
        t.eq("正确回调接受", lab.callback(location).status, 200)
        t.eq("回调重复拒绝", lab.callback(location).status, 400)
        fields = c.token_fields()
        t.eq("2.1主路径token不传redirect_uri", "redirect_uri" in fields, False)
        t.eq("code不能作为Bearer", c.call("tools/list", token=fields["code"]).status, 401)
        for name, changed in [("wrong_verifier", {"code_verifier": "Z" * 43}),
                              ("wrong_client", {"client_id": "other-client"}),
                              ("wrong_resource", {"resource": a.resource + "/other"}),
                              ("compat_redirect", {"redirect_uri": a.redirect_uri + "/wrong"})]:
            t.eq("换码拒绝-" + name, w.request("POST", a.issuer + "/token", form={**fields, **changed}).status, 400)
        extra = {**fields, "scope": "docs:read calc:use"}
        t.eq("换码不能追加scope", w.request("POST", a.issuer + "/token", form=extra).status, 400)
        t.eq("错误换码没有发token", a.counts["tokens_issued"], 0)
        grant = c.exchange()
        t.eq("正确verifier换码成功", grant["status"], 200)
        t.eq("一次发行", a.counts["tokens_issued"], 1)
        t.eq("换码后清除verifier", c.flow.verifier, "")
        token = c.access_token
        t.eq("docs权限读取成功", c.call("tools/call", name="read_doc", arguments={"doc_id": "backup"}).status, 200)
        before = a.counts["resource_executions"]
        t.eq("calc权限不足403", c.call("tools/call", name="calculate", arguments={"left":30,"right":14}).status, 403)
        t.eq("拒绝没有执行", a.counts["resource_executions"], before)
        t.eq("对象ACL仍限制", c.call("tools/call", name="read_doc", arguments={"doc_id":"bob_private"}).status, 403)
        t.raises("相同token不能用于另一个audience", lambda: a.validate_access(token, a.resource + "/other"), ProtocolError)
        t.eq("旧码重复换取拒绝", w.request("POST", a.issuer + "/token", form=fields).status, 400)
        t.eq("重放后原token撤销", c.call("tools/list", token=token).status, 401)
        old_count = a.counts["token_requests"]
        deny = lab.authorize(choice="deny")
        t.eq("拒绝是正常终态", deny["status"], "denied")
        t.eq("拒绝不换token", a.counts["token_requests"], old_count)
        # 新 grant 不修改旧 token 的 scope。
        lab.authorize("docs:read"); old_token = c.access_token
        lab.authorize("docs:read calc:use")
        t.eq("扩scope需新grant", a.validate_access(c.access_token, a.resource).scopes, SCOPES)
        t.eq("旧token没有获得新scope", a.validate_access(old_token, a.resource).scopes, frozenset({"docs:read"}))
        t.eq("operator计算16", c.call("tools/call", name="calculate", arguments={"left":30,"right":14}).json()["result"]["structuredContent"]["value"], 16)
        a.clock.advance(a.token_ttl)
        t.eq("token到期边界401", c.call("tools/list").status, 401)
        # 新授权码到期；受控时钟，无需睡眠一分钟。
        u = c.begin(); loc = lab.browser_decision(u); lab.callback(loc)
        f = c.token_fields(); a.clock.advance(a.code_ttl)
        t.eq("code到期拒绝", w.request("POST", a.issuer + "/token", form=f).status, 400)
        t.ok("HTTP审计不含任何秘密", not any(s in canonical(w.events) for s in a.secret_values))
        t.ok("审计只有方法路径状态", all(set(e)=={"method","path","status"} for e in w.events))
    # 真正并发提交同一码，只允许一个成功兑换。
    with LocalLab() as lab:
        c, a, w = lab.client, lab.authority, lab.wire
        c.discover(); u=c.begin(); loc=lab.browser_decision(u); lab.callback(loc)
        fields=c.token_fields(); results=[]; barrier=threading.Barrier(2)
        def redeem():
            barrier.wait(timeout=3)
            results.append(w.request("POST", a.issuer+"/token", form=fields).status)
        threads=[threading.Thread(target=redeem) for _ in range(2)]
        for th in threads: th.start()
        for th in threads: th.join(timeout=5)
        t.eq("并发兑换只一成功", sorted(results), [200,400])
        t.eq("并发只签发一次", a.counts["tokens_issued"], 1)
        t.ok("并发线程已退出", all(not th.is_alive() for th in threads))
    t.ok("服务监听线程已关闭", all(not th.is_alive() for th in lab.threads))
    return t.report()

In [24]:
checks = run_checks()
print(f"固定检查通过：{checks['count']}项。")
print("真实账号登录测试：", checks["real_identity_login_tested"])

固定检查通过：76项。
真实账号登录测试： False


### 保持检查不变，故意改错实现

依次把S256原始摘要改成十六进制文字、跳过issuer检查、跳过scope、重新启用已消费授权码。原检查应该失败；每次实验结束都恢复实现。

这些是本课已知错误回归，不代表穷尽了OAuth实现风险。

In [25]:
# %% 09 故障注入：改变实现，确认原有检查能发现回归

def run_mutation_checks() -> dict:
    """依次替换真实实现并运行原检查；不改变检查预期，最后恢复原函数。"""
    results = []
    original_pkce = pkce_s256
    original_callback = check_callback
    original_access = Authority.validate_access
    original_exchange = Authority.exchange

    def wrong_digest(verifier):
        original_pkce(verifier)  # 仍校验输入，但错误地编码 hexdigest 文本。
        return base64.urlsafe_b64encode(hashlib.sha256(verifier.encode("ascii")).hexdigest().encode("ascii")).rstrip(b"=").decode("ascii")

    def ignore_issuer(params, flow, now):
        return original_callback({**params, "iss": flow.issuer}, flow, now)

    def ignore_scope(self, token, audience, scope=None):
        return original_access(self, token, audience, None)

    def reuse_code(self, fields):
        with self.lock:
            row = self.codes.get(sha(fields.get("code", "")))
            if row is not None:
                row.used = False
        return original_exchange(self, fields)

    cases = [
        ("把SHA256十六进制文字当作摘要", "pkce", wrong_digest),
        ("跳过回调issuer校验", "callback", ignore_issuer),
        ("跳过工具scope校验", "access", ignore_scope),
        ("重新启用已消费的授权码", "exchange", reuse_code),
    ]
    for name, target, mutant in cases:
        try:
            if target == "pkce": globals()["pkce_s256"] = mutant
            elif target == "callback": globals()["check_callback"] = mutant
            elif target == "access": Authority.validate_access = mutant
            else: Authority.exchange = mutant
            try:
                run_checks()
            except AssertionError:
                results.append({"fault": name, "detected": True})
            else:
                raise AssertionError("测试未发现故障：" + name)
        finally:
            globals()["pkce_s256"] = original_pkce
            globals()["check_callback"] = original_callback
            Authority.validate_access = original_access
            Authority.exchange = original_exchange
    return {"count": len(results), "cases": results, "implementation_restored": True}

In [26]:
mutations = run_mutation_checks()
for item in mutations["cases"]:
    print(item["fault"], "→ 检测到：", item["detected"])
assert mutations["implementation_restored"]

把SHA256十六进制文字当作摘要 → 检测到： True
跳过回调issuer校验 → 检测到： True
跳过工具scope校验 → 检测到： True
重新启用已消费的授权码 → 检测到： True


## 16．保存实验报告，不保存凭证

下面与脚本一样建立新的时间戳目录，不覆盖旧实验。导出中没有原始access token、code、verifier、cookie或审批票据。`http_trace.json`只保留方法、路径、状态，没有完整URL查询串和HTTP头。

这只是减少日志敏感性，不是加密、安全审计存储或秘密生命周期治理。本课所有令牌和待处理状态都在内存中，退出即失效。

In [27]:
# %% 11 CLI 与无秘密的报告

def new_run_dir(root: str | Path) -> Path:
    root = Path(root)
    root.mkdir(parents=True, exist_ok=True)
    stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ")
    directory = root / ("run_" + stamp)
    directory.mkdir()
    return directory


def write_report(directory: Path, name: str, value) -> None:
    (directory/name).write_text(json.dumps(value, ensure_ascii=False, indent=2, allow_nan=False)+"\n",encoding="utf-8")


def main(argv=None) -> int:
    parser=argparse.ArgumentParser(description=__doc__)
    mode=parser.add_mutually_exclusive_group()
    mode.add_argument("--checks-only", action="store_true")
    mode.add_argument("--demo-only", action="store_true")
    mode.add_argument("--interactive", action="store_true", help="手动浏览器批准/拒绝；不是真实登录")
    parser.add_argument("--wait-seconds",type=int,default=180)
    parser.add_argument("--output-dir",default="lesson_53_outputs")
    args=parser.parse_args(argv)
    require(10 <= args.wait_seconds <= 600, "等待时间范围为10–600秒")
    if args.checks_only:
        checks=run_checks()
        print(f"固定检查通过：{checks['count']}项；真实回环HTTP，不是真实身份登录或安全认证。")
        return 0
    directory=new_run_dir(args.output_dir)
    manifest={"lesson":53,"version":VERSION,"python":platform.python_version(),
              "platform":platform.platform(),"sources":SOURCES,
              "oauth21_status_checked":"2026-11-02; draft-17", "status":"running",
              "limits":["固定Alice模拟身份","无TLS/refresh/JWT/生产限流","只有标准库本机实验","未进行官方SDK认证"]}
    write_report(directory,"manifest.json",manifest)
    try:
        if args.interactive:
            result=run_interactive(args.wait_seconds)
        else:
            if not args.demo_only:
                checks=run_checks();write_report(directory,"checks.json",checks)
                print(f"固定检查通过：{checks['count']}项。")
            result=run_demo()
            if not args.demo_only:
                write_report(directory,"mutations.json",run_mutation_checks())
        write_report(directory,"experiment.json",result)
        if "wire" in result:
            write_report(directory,"http_trace.json",result["wire"])
        manifest["status"]="completed"
        print(canonical({k:v for k,v in result.items() if k not in {"wire","limits"}}))
        print("报告目录：", directory.resolve())
    except BaseException as error:
        manifest["status"]="failed"
        manifest["error_type"]=type(error).__name__  # 不记录可能含 code/verifier 的异常全文。
        raise
    finally:
        write_report(directory,"manifest.json",manifest)
    return 0

In [28]:
output_dir = new_run_dir("lesson_53_outputs")
write_report(output_dir, "checks.json", checks)
write_report(output_dir, "experiment.json", experiment)
write_report(output_dir, "http_trace.json", experiment["wire"])
write_report(output_dir, "mutations.json", mutations)
write_report(output_dir, "manifest.json", {
    "lesson": 53, "version": VERSION, "python": platform.python_version(),
    "mode": "notebook_loopback_http", "status": "completed", "sources": SOURCES,
    "oauth21_status_checked": "2026-11-02; draft-17",
    "real_login_tested": False, "tls_tested": False,
})
print("报告目录：", output_dir.resolve())

报告目录： /mnt/data/_lesson53_validation/notebook_run/lesson_53_outputs/run_20261102T131402_978802Z


## 17．终端运行与验收

```bash
# 只运行自动检查
python lesson_53_oauth_authorization_code_and_pkce.py --checks-only

# 完整演示、检查、故障注入并保存
python lesson_53_oauth_authorization_code_and_pkce.py

# 只跑HTTP演示
python lesson_53_oauth_authorization_code_and_pkce.py --demo-only

# 在本机浏览器批准或拒绝
python lesson_53_oauth_authorization_code_and_pkce.py --interactive
```

请能够解释：

1. 为什么code不能直接作为Bearer使用，错误verifier也不能换token？
2. 为什么state、issuer、PKCE、audience和scope不能只选一个检查？
3. 为什么第二次获得新权限，旧token仍然只读？
4. 为什么用户拒绝后不应该调用token端点？
5. 为什么客户端不把token交换当作可盲目重试的幂等请求？
6. 为什么这份有真实HTTP的演示仍不是真实用户登录或生产OAuth实现？

**下一课：令牌生命周期、刷新轮换与撤销。** 将继续区分“拿到一次访问权限”和“长期安全保持访问权限”，不会通过把token永久保存来绕过有效期。

## 原始资料与范围

- **[R1]** [OAuth 2.1 当前状态](https://datatracker.ietf.org/doc/draft-ietf-oauth-v2-1/) 与 [固定 draft-17](https://www.ietf.org/ietf-ftp/internet-drafts/draft-ietf-oauth-v2-1-17.html)，尤其 §10.2：草案状态、public client、授权码及token请求变化。查询日期2026-11-02。
- **[R2]** [RFC 7636](https://www.rfc-editor.org/rfc/rfc7636.html)：PKCE、S256、长度和附录B公开向量。
- **[R3]** [RFC 9700](https://www.rfc-editor.org/rfc/rfc9700.html)：OAuth安全最佳实践、PKCE降级与CSRF边界。
- **[R4]** [RFC 8414](https://www.rfc-editor.org/rfc/rfc8414.html)：授权服务器metadata。
- **[R5]** [RFC 8707](https://www.rfc-editor.org/rfc/rfc8707.html)：resource indicators与目标资源绑定。
- **[R6]** [RFC 9207](https://www.rfc-editor.org/rfc/rfc9207.html)：回调issuer检查，错误响应也适用。
- **[R7]** [RFC 8252](https://www.rfc-editor.org/rfc/rfc8252.html)：公共原生客户端、外部浏览器与loopback回调例外。
- **[R8]** [MCP固定版授权规范](https://modelcontextprotocol.io/specification/2026-07-28/basic/authorization) 与 [RFC 9728](https://www.rfc-editor.org/rfc/rfc9728.html)：资源metadata与授权发现。
- [Python http.server](https://docs.python.org/3/library/http.server.html)：标准库教学HTTP服务，不建议直接用于生产。

本课没有完整TLS、真实身份认证、JWT签名校验、DPoP/mTLS、refresh token、真实OAuth供应商、生产限流、持久令牌库或官方互操作认证；也不绕过任何真实账号登录。不要把它暴露到公网或录入真实密码。